# 01 — Exploración de datos

Análisis exploratorio del dataset de cabecera del gasoducto: un mes de operación normal con muestreo cada 1 minuto, presión (P) y caudal (Q).

**Objetivo:** caracterizar la calidad y estructura de los datos, y derivar empíricamente las propiedades que justifican el diseño del detector en capas (sanidad, suavizado, divergencia P-Q, CUSUM).

**Dataset:** `data/PLOT_TS_P_Q.csv` (no versionado en el repo).

In [5]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Configuración de matplotlib
%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 100

# Ruta al CSV (relativa a la ubicación del notebook)
DATA_PATH = Path("..") / "data" / "PLOT_TS_P_Q.csv"
assert DATA_PATH.exists(), f"No se encuentra el CSV en {DATA_PATH.resolve()}"

print(f"CSV encontrado en: {DATA_PATH.resolve()}")
print(f"Tamaño: {DATA_PATH.stat().st_size / 1024:.1f} KB")

CSV encontrado en: C:\Users\agust\Portfolio\monitor-gasoducto\data\PLOT_TS_P_Q.csv
Tamaño: 1634.1 KB


## 1. Carga y primer vistazo

Lectura del CSV con separador `;` y separador decimal `.`. Parseo de la columna `TS` con formato `dd/mm/yyyy HH:MM:SS`. Inspección inicial de estructura, tipos y rango.

In [6]:
# Carga del CSV
df = pd.read_csv(
    DATA_PATH,
    sep=";",
    parse_dates=["TS"],
    date_format="%d/%m/%Y %H:%M:%S",
)

# Primer vistazo
print(f"Shape: {df.shape}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nRango temporal: {df['TS'].min()}  →  {df['TS'].max()}")
print(f"\nPrimeras 5 filas:")
df.head()

Shape: (43155, 3)

Dtypes:
TS    datetime64[us]
P            float64
Q            float64
dtype: object

Rango temporal: 2026-04-09 13:47:00  →  2026-05-09 13:46:55

Primeras 5 filas:


,TS,P,Q
0,2026-04-09 13:47:00,92.74138,850.4634
1,2026-04-09 13:48:00,92.74138,850.4634
2,2026-04-09 13:49:00,92.74138,850.4634
3,2026-04-09 13:50:00,92.74138,850.4634
4,2026-04-09 13:51:00,92.74138,850.4634


## 2. Sanidad temporal

Verificación de propiedades estructurales de la serie temporal: orden, duplicados, gaps en el muestreo y valores faltantes en P/Q. Estos chequeos son precondición de todo el pipeline del detector (suavizado, derivadas, CUSUM asumen secuencia ordenada y regular).

In [7]:
# 2.1 — ¿Está ordenado cronológicamente?
is_sorted = df["TS"].is_monotonic_increasing
print(f"TS ordenado cronológicamente: {is_sorted}")

# 2.2 — ¿Hay duplicados de TS?
n_duplicados = df["TS"].duplicated().sum()
print(f"Timestamps duplicados: {n_duplicados}")

# 2.3 — ¿NaN en P o Q?
print(f"\nNaN por columna:")
print(df[["P", "Q"]].isna().sum())

# 2.4 — Análisis de la cadencia (intervalos entre muestras consecutivas)
deltas = df["TS"].diff().dropna()
print(f"\nEstadísticas de los intervalos entre muestras:")
print(f"  Mínimo:  {deltas.min()}")
print(f"  Máximo:  {deltas.max()}")
print(f"  Mediana: {deltas.median()}")
print(f"  Moda:    {deltas.mode().iloc[0]}")

# 2.5 — Distribución de los intervalos (cuántos son "anómalos")
print(f"\nIntervalos únicos y sus frecuencias (top 10):")
print(deltas.value_counts().head(10))

# 2.6 — ¿Cuántas muestras faltantes implican los gaps?
intervalo_esperado = pd.Timedelta(minutes=1)
muestras_faltantes = ((deltas - intervalo_esperado) / intervalo_esperado).clip(lower=0).sum()
print(f"\nMuestras faltantes (estimadas a partir de gaps): {int(muestras_faltantes)}")

TS ordenado cronológicamente: True
Timestamps duplicados: 0

NaN por columna:
P    0
Q    0
dtype: int64

Estadísticas de los intervalos entre muestras:
  Mínimo:  0 days 00:01:00
  Máximo:  0 days 00:46:55
  Mediana: 0 days 00:01:00
  Moda:    0 days 00:01:00

Intervalos únicos y sus frecuencias (top 10):
TS
0 days 00:01:00    43153
0 days 00:46:55        1
Name: count, dtype: int64

Muestras faltantes (estimadas a partir de gaps): 45


In [8]:
# Localización del gap único en el tiempo
intervalo_esperado = pd.Timedelta(minutes=1)
deltas = df["TS"].diff()

# Filas posteriores a un gap (delta > 1 min)
idx_post_gap = df.index[deltas > intervalo_esperado]

print(f"Cantidad de gaps detectados: {len(idx_post_gap)}")

for idx in idx_post_gap:
    ts_pre = df.loc[idx - 1, "TS"]
    ts_post = df.loc[idx, "TS"]
    duracion = ts_post - ts_pre

    # Posición relativa dentro del rango total
    ts_inicio = df["TS"].iloc[0]
    ts_fin = df["TS"].iloc[-1]
    pos_relativa = (ts_pre - ts_inicio) / (ts_fin - ts_inicio) * 100

    print(f"\nGap detectado:")
    print(f"  Último dato pre-gap : {ts_pre}")
    print(f"  Primer dato post-gap: {ts_post}")
    print(f"  Duración del gap    : {duracion}")
    print(f"  Posición relativa   : {pos_relativa:.1f}% del mes")

Cantidad de gaps detectados: 1

Gap detectado:
  Último dato pre-gap : 2026-05-09 13:00:00
  Primer dato post-gap: 2026-05-09 13:46:55
  Duración del gap    : 0 days 00:46:55
  Posición relativa   : 99.9% del mes
